In [1]:
import os
import sys
import getpass
from pathlib import Path
import google.generativeai as genai

# ============================================================================
# SECURE KEY MANAGER (add this function)
# ============================================================================
_SECURE_KEYS = {}

def secure_get_key(service_name, key_name, prompt_message=None):
    """Get API key securely - stored only in memory"""
    global _SECURE_KEYS
    
    # Return from memory if already entered
    if key_name in _SECURE_KEYS:
        return _SECURE_KEYS[key_name]
    
    # Prompt user for key
    if prompt_message is None:
        prompt_message = f"Enter your {service_name} API key: "
    
    print(f"\n🔐 {service_name} API Key Required")
    print("-" * 50)
    key = getpass.getpass(prompt_message)
    
    if not key:
        raise ValueError(f"No API key provided for {service_name}")
    
    # Store in memory only
    _SECURE_KEYS[key_name] = key
    
    print(f"✓ {service_name} API key loaded (memory only)")
    return key

# ============================================================================
# GEMINI SETUP (using secure key instead of .env)
# ============================================================================

# Get API key securely (prompts if not in memory)
GEMINI_API_KEY = secure_get_key("Gemini", "GEMINI_API_KEY")

# Configure Gemini
genai.configure(api_key=GEMINI_API_KEY)
print(f"✓ Gemini API configured")

# Model selection
MODEL_NAME = input("\nEnter Gemini model (press Enter for 'gemma-4-31b-it'): ").strip()
if not MODEL_NAME:
    MODEL_NAME = "gemma-4-31b-it"

# Initialize model
MODEL = genai.GenerativeModel(MODEL_NAME)
print(f"✓ Model initialized: {MODEL_NAME}")


🔐 Gemini API Key Required
--------------------------------------------------


ValueError: No API key provided for Gemini

In [2]:
# ============================================================================
# OPTIONAL: DEEPSEEK API SETUP (SECURE VERSION)
# ============================================================================

from openai import OpenAI

# Get DeepSeek API key securely
DEEPSEEK_API_KEY = secure_get_key("DeepSeek", "DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = input("\nDeepSeek base URL (press Enter for default): ").strip()
if not DEEPSEEK_BASE_URL:
    DEEPSEEK_BASE_URL = "https://api.deepseek.com"

# Initialize DeepSeek client
deepseek_client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

print(f"✓ DeepSeek client configured")
print(f"  Base URL: {DEEPSEEK_BASE_URL}")

# Optional: Wrapper function
# def deepseek_generate(prompt: str, temperature: float = 0.7, max_tokens: int = 1500) -> str:
#     """Generate content using DeepSeek API"""
#     try:
#         rate_limited_api_call()
#         response = deepseek_client.chat.completions.create(
#             model="deepseek-chat",
#             messages=[{"role": "user", "content": prompt}],
#             temperature=temperature,
#             max_tokens=max_tokens
#         )
#         return response.choices[0].message.content
#     except Exception as e:
#         print(f"⚠ DeepSeek error: {e}")
#         return ""


🔐 DeepSeek API Key Required
--------------------------------------------------
✓ DeepSeek API key loaded (memory only)
✓ DeepSeek client configured
  Base URL: https://api.deepseek.com


In [3]:
def prompt_taker(prompt: str, model_name: str = None, temperature: float = 0.7, max_tokens: int = 2048) -> dict:
    """
    Send a prompt to the LLM and receive the response.
    
    Parameters:
    -----------
    prompt : str
        The prompt to send to the model
    model_name : str
        Model to use (default: will use configured MODEL_NAME)
    temperature : float
        Temperature for generation (0.0-1.0)
    max_tokens : int
        Maximum tokens in response
    
    Returns:
    --------
    dict : {
        'prompt': str,
        'response': str,
        'model': str,
        'temperature': float,
        'status': 'success' or 'error',
        'error': str (if error)
    }
    """
    
    if model_name is None:
        model_name = MODEL_NAME
    
    print("=" * 70)
    print("PROMPT TAKER & RECEIVER")
    print("=" * 70)
    
    # Display prompt
    print(f"\n📝 PROMPT ({len(prompt)} chars):")
    print("-" * 70)
    print(prompt)
    print("-" * 70)
    
    # Display configuration
    print(f"\n⚙️  Configuration:")
    print(f"   Model: {model_name}")
    print(f"   Temperature: {temperature}")
    print(f"   Max tokens: {max_tokens}")
    
    try:
        # Send to model
        print(f"\n🔄 Sending to API...")
        
        # Modified call for your prompt_taker function
        response = MODEL.generate_content(
    prompt,
    generation_config=genai.types.GenerationConfig(
        temperature=0.0, 
        max_output_tokens=5,
        # We pick the 5 most likely "rambling" triggers
        stop_sequences=["\n", "*", "Calculation", "Target", "Input"]
    )
)
        
        response_text = response.text
        
        # Display response
        print(f"\n✅ RESPONSE ({len(response_text)} chars):")
        print("-" * 70)
        print(response_text)
        print("-" * 70)
        
        # Display stats
        print(f"\n📊 Statistics:")
        print(f"   Prompt length: {len(prompt)} chars")
        print(f"   Response length: {len(response_text)} chars")
        print(f"   Ratio: {len(response_text)/len(prompt):.2f}x")
        
        return {
            'prompt': prompt,
            'response': response_text,
            'model': model_name,
            'temperature': temperature,
            'max_tokens': max_tokens,
            'prompt_length': len(prompt),
            'response_length': len(response_text),
            'status': 'success',
            'error': None
        }
        
    except Exception as e:
        error_msg = str(e)
        print(f"\n❌ ERROR: {error_msg}")
        
        return {
            'prompt': prompt,
            'response': None,
            'model': model_name,
            'temperature': temperature,
            'status': 'error',
            'error': error_msg
        }


def multi_prompt_test(prompts: list, model_name: str = None, temperature: float = 0.7) -> list:
    """
    Test multiple prompts and compare responses.
    
    Parameters:
    -----------
    prompts : list of dicts
        Each dict should have: {'name': str, 'prompt': str}
    model_name : str
        Model to use (default: will use configured MODEL_NAME)
    temperature : float
        Temperature for generation
    
    Returns:
    --------
    list : Results for each prompt
    """
    
    if model_name is None:
        model_name = MODEL_NAME
    
    results = []
    
    for i, prompt_data in enumerate(prompts, 1):
        print(f"\n\n{'#' * 70}")
        print(f"# TEST {i}/{len(prompts)}: {prompt_data.get('name', f'Prompt {i}')}")
        print(f"{'#' * 70}")
        
        result = prompt_taker(
            prompt=prompt_data['prompt'],
            model_name=model_name,
            temperature=temperature
        )
        results.append(result)
        
        print(f"\n⏳ Waiting before next prompt...")
        import time
        time.sleep(0.5)
    
    return results


print("\n✓ Prompt taker/receiver functions loaded!")
print("   - prompt_taker(prompt) : Send single prompt and get response")
print("   - multi_prompt_test(prompts) : Test multiple prompts with comparison")


✓ Prompt taker/receiver functions loaded!
   - prompt_taker(prompt) : Send single prompt and get response
   - multi_prompt_test(prompts) : Test multiple prompts with comparison


In [4]:
# ============================================================================
# ONE-WORD PROMPT TEST - STRUCTURED TESTING
# ============================================================================

ONE_WORD_PROMPT =VALUE_PROMPT_CODEACT = """<start_of_turn>user
Numbers: {input}
Target: 24

ANSWER WITH ONE WORD ONLY - NO EXPLANATION, NO WORKING, NO BULLET POINTS:
Answer "sure" if reachable. Answer "likely" if closest is 20-24. Answer "impossible" otherwise.

Your word:
<end_of_turn>
<start_of_turn>model
"""

# Test cases with expected outcomes


print("=" * 70)
print("ONE-WORD PROMPT STRUCTURE TEST")
print("=" * 70)
print(f"\nTest framework ready with {len(test_cases)} test cases")
print(f"\nPrompt size: {len(ONE_WORD_PROMPT)} chars")
print(f"Temperature: 0.5 (lower = more consistent)")
print(f"Max tokens: 50 (force brevity)")
print(f"\nTest cases:")
for i, tc in enumerate(test_cases, 1):
    print(f"  {i}. {tc['name']:<25} → Expected: {tc['expected']:<12} ({tc['reason']})")


ONE-WORD PROMPT STRUCTURE TEST


NameError: name 'test_cases' is not defined

In [5]:
# ============================================================================
# MANUAL INPUT - TEST ONE-WORD PROMPT WITH YOUR OWN NUMBERS
# ============================================================================

ONE_WORD_PROMPT = VALUE_PROMPT_CODEACT = """<start_of_turn>user
Numbers: {input}
Target: 24

ANSWER WITH ONE WORD ONLY - NO EXPLANATION, NO WORKING, NO BULLET POINTS:
Answer "sure" if reachable. Answer "likely" if close to 20-24. Answer "impossible" otherwise.

Your word:
<end_of_turn>
<start_of_turn>model
"""
# Add the <|think|> control token logic for Gemma 4
# ONE_WORD_PROMPT = """<start_of_turn>user
# Item: {input}
# Map to exactly one: [sure, likely, impossible]
# Constraint: No reasoning. No math.
# Result: <end_of_turn>
# <start_of_turn>model
# """

# ============================================================================
# ENTER YOUR NUMBERS HERE (as a list or comma-separated)
# ============================================================================

# Example: [1, 2, 4, 7] or [15, 9] or [19, 6, 9]
your_numbers = [90, 6, 9]  # ← CHANGE THIS TO YOUR NUMBERS

print("=" * 70)
print("MANUAL ONE-WORD PROMPT TEST")
print("=" * 70)
print(f"\nYour numbers: {your_numbers}")
print(f"\nPrompt:")
print("-" * 70)
print(ONE_WORD_PROMPT.format(input=str(your_numbers)))
print("-" * 70)

# Ready to test
print(f"\n✓ Ready to send to LLM")
print(f"  Press the 'Run' button below to test with these numbers")
print(f"\n  To change numbers: Edit 'your_numbers = [...]' above and run again")


MANUAL ONE-WORD PROMPT TEST

Your numbers: [90, 6, 9]

Prompt:
----------------------------------------------------------------------
<start_of_turn>user
Numbers: [90, 6, 9]
Target: 24

ANSWER WITH ONE WORD ONLY - NO EXPLANATION, NO WORKING, NO BULLET POINTS:
Answer "sure" if reachable. Answer "likely" if close to 20-24. Answer "impossible" otherwise.

Your word:
<end_of_turn>
<start_of_turn>model

----------------------------------------------------------------------

✓ Ready to send to LLM
  Press the 'Run' button below to test with these numbers

  To change numbers: Edit 'your_numbers = [...]' above and run again


In [6]:
# ============================================================================
# SEND TO LLM (uses your_numbers from previous cell)
# ============================================================================

# Format the prompt
numbers_str = str(your_numbers)
prompt = ONE_WORD_PROMPT.format(input=numbers_str)

print("=" * 70)
print("SENDING ONE-WORD PROMPT TO LLM")
print("=" * 70)
print(f"\nYour numbers: {your_numbers}")
print(f"Settings: temperature=0.5, max_tokens=50 (force brevity)")

# Send to model
result = prompt_taker(
    prompt=prompt,
    model_name=MODEL_NAME,
    temperature=0,
    max_tokens=5
)

# Analyze response
if result['status'] == 'success':
    response = result['response'].strip().lower()
    
    print(f"\n" + "=" * 70)
    print("RESPONSE ANALYSIS")
    print("=" * 70)
    
    # Extract judgment
    judgment = None
    for word in ['sure', 'likely', 'impossible']:
        if word in response:
            judgment = word
            break
    
    if judgment is None:
        judgment = response.split()[-1] if response.split() else 'unclear'
    
    print(f"\n📊 JUDGMENT: {judgment.upper()}")
    print(f"\n   ✓ sure      = You found a solution")
    print(f"   ✓ likely    = Closest is 20-24")
    print(f"   ✓ impossible = Closest is far from 24")
    
    print(f"\n📈 METRICS:")
    print(f"   Response length: {len(response)} chars")
    print(f"   Prompt length: {len(prompt)} chars")
    print(f"   Ratio: {len(response)/len(prompt):.2f}x (target: <2.0x)")
    
    print(f"\n📝 FULL RESPONSE:")
    print("-" * 70)
    print(response)
    print("-" * 70)
    
else:
    print(f"\n❌ Error: {result['error']}")


SENDING ONE-WORD PROMPT TO LLM

Your numbers: [90, 6, 9]
Settings: temperature=0.5, max_tokens=50 (force brevity)


NameError: name 'MODEL_NAME' is not defined

In [6]:
# Please install OpenAI SDK first: `pip3 install openai`
import os
from openai import OpenAI

client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url="https://api.deepseek.com")

# response = client.chat.completions.create(
#     model="deepseek-chat",
#     messages=[
#         {"role": "system", "content": "You are a math reasoning assistant. Produce at most 3 short steps of reasoning, then output 'ANSWER: sure/likely/impossible'."},
#         {"role": "user", "content": """Numbers: [2,2,7] Target: 24

# Think step by step (max 3 short bullet points):
# - Try combinations: (2*2)*7=28, (7*2)+2=16, (7-2)*2=10.
# - 28 is within 20-28 but not 24.
# - No combination gives exactly 24.

# ANSWER: likely"""}
#     ],
#     temperature=0,
#     max_tokens=150  # forces concise output
# )



In [10]:
def solve_24_game(numbers, target):
    client = OpenAI(api_key="sk-b686359b5bc842129c35400deadece97", base_url="https://api.deepseek.com")
    prompt = f"""Numbers: {numbers} Target: {target}

Think step by step (max 5 short bullet points):
- Try combinations: (a*b)*c, (a+b)*c, etc. Learn from the combinations. For example, if one result is too far off or close, adjust for the next approach accordingly. You can make 5 attempts.
- Check if any result equals {target}.
- If not, check if any result is between {target-4} and {target+4}.

ANSWER(strictly use one word in a new line): sure/likely(if within 20-28 range, don't print this in final answer)/impossible"""
    adaptive_prompt = f"""Numbers: {numbers} Target: {target}

You are a clever math puzzler. Make at most 5 attempts. After each attempt, note if the result is too high, too low, or close. Show your reasoning in short plain‑text bullet points. Do not use LaTeX.

Examples:

[3,6,10]:
- Attempt 1: 10 + 6 = 16, then 16 + 3 = 19 (too low).
- Attempt 2: 10 * 3 = 30, then 30 - 6 = 24 (exact!).
ANSWER: sure

[2,2,7]:
- Attempt 1: 7 * 2 = 14, then 14 + 2 = 16 (too low).
- Attempt 2: 7 * 2 = 14, then 14 * 2 = 28 (too high, within 20-28).
- Attempt 3: 7 + 2 = 9, then 9 * 2 = 18 (still low). Best is 28 within range.
ANSWER: likely

[10,10,10]:
- Attempt 1: 10 + 10 = 20, then 20 + 10 = 30 (too high).
- Attempt 2: 10 + 10 = 20, then 20 - 10 = 10 (too low).
- Attempt 3: 10 * 10 = 100, then 100 / 10 = 10 (still low).
- Attempt 4: 10 * 10 = 100, then 100 - 10 = 90 (far high).
All results are 10, 30, or 90. None between 20-28.
ANSWER: impossible

Now solve for {numbers} with target {target}. Use the same style.

IMPORTANT DECISION RULE (apply after your reasoning):
- If you found a result **within 0.001 of {target}** (i.e., {target-0.001} to {target+0.001}) → ANSWER: sure
- Else if you found any result between 20 and 28 (inclusive) → ANSWER: likely
- Else → ANSWER: impossible

After reasoning, write a blank line then "ANSWER: sure/likely/impossible"
"""
    
    response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=[
            {"role": "system", "content": "You are a math reasoning assistant. Produce at most 5 short steps, then 'ANSWER: ...'."},
            {"role": "user", "content": adaptive_prompt}
        ],
        extra_body={"thinking": {"type": "disabled"}},
        temperature=0,
        max_tokens=300
    )
    return response.choices[0].message.content

In [11]:
print(solve_24_game([2,2,7], 24))   # works
print(solve_24_game([3,3,8], 24))   # adapts
print(solve_24_game([90,6,9], 24))   # new case
print(solve_24_game([100, 4, 1], 24))   # impossible case
print(solve_24_game([75, 3, 1], 24))   # impossible case
print(solve_24_game([100, 5, 1], 24))   # impossible case
print(solve_24_game([3, 3, 9], 24))   # impossible case
print(solve_24_game([3, 6, 10], 24))   # impossible case
print(solve_24_game([1, 2, 4, 7], 24))   # sure case
print(solve_24_game([10, 1,2.18191], 24))   # sure case



- Attempt 1: 7 * 2 = 14, then 14 + 2 = 16 (too low).  
- Attempt 2: 7 * 2 = 14, then 14 * 2 = 28 (too high, but within 20-28).  
- Attempt 3: 7 + 2 = 9, then 9 * 2 = 18 (still low).  
- Attempt 4: 7 * 2 = 14, then 14 * 2 = 28 (best so far, within range).  

ANSWER: likely
- Attempt 1: 8 * 3 = 24, then 24 + 3 = 27 (too high, but within 20-28).
- Attempt 2: 8 * 3 = 24, then 24 - 3 = 21 (too low, but within 20-28).
- Attempt 3: 8 + 3 = 11, then 11 * 3 = 33 (too high, outside 20-28).
- Attempt 4: 8 * 3 = 24, then 24 * 3 = 72 (far too high).
- Attempt 5: 8 * 3 = 24, then 24 / 3 = 8 (too low).

Best results are 27 and 21, both within 20-28.

ANSWER: likely
- Attempt 1: 90 / 6 = 15, then 15 + 9 = 24 (exact!).
ANSWER: sure
- Attempt 1: 100 / 4 = 25, then 25 - 1 = 24 (exact!).
ANSWER: sure
- Attempt 1: 75 / 3 = 25, then 25 - 1 = 24 (exact!).
ANSWER: sure
- Attempt 1: 100 / 5 = 20, then 20 + 1 = 21 (too low, but within 20-28).
- Attempt 2: 100 / 5 = 20, then 20 * 1 = 20 (still low).
- Attempt 3: